# 03 preprocessing features

essentiel imports

In [1]:
from classification.preprocessing import remove_invalid_targets
from classification.validation import NUMERIC_RULES, CATEGORY_RULES, invalid_category
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

load dataframe

In [2]:
df = pd.read_csv("../data/raw/student_mental_health.csv")

splitting data to avoid data leakage

In [3]:
from classification.model_config import RANDOM_STATE
from classification.validation import invalid_category
cleaned_df = df.loc[ (df["Mental_Health_Risk"].notna()) & ~invalid_category(df["Mental_Health_Risk"], CATEGORY_RULES["Mental_Health_Risk"])].copy()

X = cleaned_df.drop(columns = ["Student_ID", "Mental_Health_Risk"])

risk_codes = {"Low": 0, "Medium": 1, "High": 2}
y = cleaned_df["Mental_Health_Risk"].map(risk_codes)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)


casting

In [4]:
transformed = X_train.copy()
for column in NUMERIC_RULES:
    transformed[column] = pd.to_numeric(
        transformed[column], errors="coerce"
    ).astype("float64")

for column in ["Student_ID", *CATEGORY_RULES]:
    if column in transformed.columns:
        text = transformed[column].astype("string")
        transformed[column] = text.astype(object).where(text.notna(), np.nan)

unlogic values to NAN

In [5]:
from classification.validation import get_invalid_masks


for column , invalid in get_invalid_masks(transformed).items():
    transformed[column] = transformed[column].mask(invalid, np.nan)


## Missing-value handling

we made the impossible values and the social media / screen time contradictions NaN.
for this first baseline we use median for numerical columns and most frequent for categorical columns.
we fit the imputer on training rows only.

In [13]:
from classification.preprocessing import impute_missing_values

print("missing before imputation:")
print(transformed.isna().sum()[transformed.isna().sum() > 0])

imputer = impute_missing_values()
imputed_train = imputer.fit_transform(transformed)

print("missing after imputation:")
print(imputed_train.isna().sum().sum())
display(imputed_train.head())

missing before imputation:
Sleep_Hours           49
GPA                   27
Social_Media_Hours    74
Screen_Time_Hours     25
Financial_Stress      71
Family_Support        40
Attendance_Rate       34
Diet_Quality          53
dtype: int64
missing after imputation:
0


,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate,Gender,Physical_Activity,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History
521,22.0,1.0,6.7,11.5,2.28,5.9,8.4,5.5,6.5,81.6,Female,Moderate,No,Average,No
1393,21.0,1.0,8.1,12.0,2.00,5.2,7.9,7.2,4.3,77.1,Female,Moderate,Yes,Poor,Yes
1710,23.0,4.0,6.2,24.2,2.45,6.1,9.6,6.4,7.8,96.1,Male,Low,Yes,Good,No
555,20.0,2.0,7.2,19.3,2.48,6.4,8.2,4.8,5.4,78.9,Male,High,Yes,Average,No
523,22.0,4.0,6.4,20.8,2.16,5.6,8.5,8.4,8.7,62.5,Female,Moderate,No,Good,Yes


## Scaling and transformations

now we use StandardScaler on the numerical columns after imputation.
the categorical columns pass through without scaling. we do not cap the IQR outliers because the EDA found them plausible.

In [16]:
from classification.preprocessing import scale_numeric_columns

scaler = scale_numeric_columns()
scaled_train = scaler.fit_transform(imputed_train)
display(scaled_train[list(NUMERIC_RULES)].agg(["min", "max", "mean"]).round())

,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate
min,-2.0,-1.0,-3.0,-3.0,-3.0,-3.0,-3.0,-2.0,-3.0,-4.0
max,3.0,2.0,3.0,3.0,3.0,4.0,3.0,2.0,2.0,2.0
mean,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0


## Categorical encoding

Gender has no order, so we use one hot encoding.
Physical_Activity and Diet_Quality have an order, so we use ordinal encoding.
Part_Time_Job and Previous_Mental_Health_History become 0 for No and 1 for Yes.
the target is already in `y`, so it is not encoded here.

In [17]:
from classification.preprocessing import categorical_encode

encoder = categorical_encode()
encoded_train = encoder.fit_transform(scaled_train)

print("shape before encoding:", scaled_train.shape)
print("shape after encoding:", encoded_train.shape)
print("encoded columns:", encoder.get_feature_names_out())

shape before encoding: (1600, 15)
shape after encoding: (1600, 17)
encoded columns: ['onehot__Gender_Female' 'onehot__Gender_Male' 'onehot__Gender_Non-binary'
 'ordinal__Physical_Activity' 'ordinal__Diet_Quality'
 'binary__Part_Time_Job' 'binary__Previous_Mental_Health_History'
 'remainder__Age' 'remainder__Year_of_Study' 'remainder__Sleep_Hours'
 'remainder__Study_Hours_Per_Week' 'remainder__GPA'
 'remainder__Social_Media_Hours' 'remainder__Screen_Time_Hours'
 'remainder__Financial_Stress' 'remainder__Family_Support'
 'remainder__Attendance_Rate']


## Feature engineering

Student_ID is removed because it only identifies a student. we keep the other measured features for the first baseline and do not create new features without testing whether they help.

## ColumnTransformer and Pipeline design

this notebook shows imputation, scaling, then encoding on training rows so we can inspect each result.
in the training notebook we pass raw `X_train` to a model pipeline. cross-validation fits a new preprocessor inside each training fold; the test set stays untouched until final evaluation.

In [19]:
from classification.preprocessing import build_preprocessor

preprocessor = build_preprocessor()
print("pipeline steps:", list(preprocessor.named_steps))


pipeline steps: ['cast_columns', 'invalid_to_nan', 'imputer', 'scaler', 'encode']
